<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 14


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Supplier в C#, который будет представлять информацию о поставщиках товаров или услуг. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Supplier:


Атрибуты: ID поставщика (SupplierId), Название компании (CompanyName), Тип продукции (ProductType).


Методы:


GetCompanyInfo(): метод для получения информации о компании.


ProvideQuote(): метод для предоставления котировки на товары или услуги.


SubmitOrder(): метод для отправки заказа поставщику.
Требования к производным классам:

1. Производитель (Manufacturer): Должен содержать дополнительные атрибуты, такие как Год основания (FoundedYear). Метод ProvideQuote() должен быть переопределен для включения информации о годе основания компании в котировку.
2. Ритейлер (Retailer): Должен содержать дополнительные атрибуты, такие как Расположение магазина (StoreLocation). Метод SubmitOrder() должен быть переопределен для добавления информации о расположении магазина при отправке заказа.
3. Импортер (Importer) (если требуется третий класс): Должен содержать дополнительные атрибуты, такие как Страна происхождения товара (OriginCountry). Метод GetCompanyInfo() должен быть переопределен для отображения страны происхождения товара вместе с остальной информацией о компании.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [9]:
using System;
using System.Collections.Generic;
using System.Linq;

// Делегаты
public delegate void OrderEventHandler(Supplier supplier, OrderEventArgs e);
public delegate void QuoteEventHandler(Supplier supplier, decimal amount);
public delegate bool SupplierFilter(Supplier supplier);

// Аргументы события заказа
public class OrderEventArgs : EventArgs
{
    public string OrderNumber { get; }
    public decimal OrderAmount { get; }
    public DateTime OrderDate { get; }
    public string ProductDetails { get; }

    public OrderEventArgs(string orderNumber, decimal orderAmount, DateTime orderDate, string productDetails)
    {
        OrderNumber = orderNumber;
        OrderAmount = orderAmount;
        OrderDate = orderDate;
        ProductDetails = productDetails;
    }
}

// Базовый класс Supplier с событиями
public class Supplier
{
    // События
    public event OrderEventHandler OnOrderSubmitted;
    public event QuoteEventHandler OnQuoteProvided;
    public event EventHandler<string> OnSupplierStatusChanged;

    // Существующие свойства
    public int SupplierId { get; set; }
    public string CompanyName { get; set; }
    public string ProductType { get; set; }

    // Новые свойства
    public decimal Rating { get; set; }
    public bool IsActive { get; set; }
    public DateTime RegistrationDate { get; set; }
    public List<string> Certifications { get; set; }
    public decimal MinimumOrderAmount { get; set; }

    // Коллекция для хранения истории заказов
    protected List<OrderEventArgs> OrderHistory = new List<OrderEventArgs>();

    public Supplier(int supplierId, string companyName, string productType)
    {
        SupplierId = supplierId;
        CompanyName = companyName;
        ProductType = productType;
        Rating = 5.0m;
        IsActive = true;
        RegistrationDate = DateTime.Now;
        Certifications = new List<string>();
        MinimumOrderAmount = 1000m;
    }

    // Существующие методы
    public virtual string GetCompanyInfo()
    {
        return $"ID: {SupplierId}, Компания: {CompanyName}, Тип продукции: {ProductType}";
    }

    public virtual string ProvideQuote()
    {
        var quoteAmount = CalculateQuote();
        RaiseQuoteProvided(quoteAmount);
        return $"Компания {CompanyName} предоставляет котировку на продукцию типа {ProductType}.";
    }

    public virtual string SubmitOrder()
    {
        var order = CreateOrder();
        OrderHistory.Add(order);
        RaiseOrderSubmitted(order);
        return $"Заказ отправлен в компанию {CompanyName}.";
    }

    // Новые методы
    public virtual decimal CalculateQuote()
    {
        return MinimumOrderAmount * 1.1m; // 10% надбавка
    }

    public virtual OrderEventArgs CreateOrder()
    {
        return new OrderEventArgs(
            $"ORD-{SupplierId}-{DateTime.Now:yyyyMMdd-HHmmss}",
            CalculateQuote(),
            DateTime.Now,
            $"Продукция типа {ProductType}"
        );
    }

    public void AddCertification(string certification)
    {
        Certifications.Add(certification);
        RaiseStatusChanged($"Добавлена сертификация: {certification}");
    }

    public virtual string GetCertifications()
    {
        return Certifications.Count > 0 
            ? $"Сертификации: {string.Join(", ", Certifications)}"
            : "Сертификации отсутствуют";
    }

    public int GetOrderCount()
    {
        return OrderHistory.Count;
    }

    public decimal GetTotalRevenue()
    {
        return OrderHistory.Sum(order => order.OrderAmount);
    }

    public virtual void UpdateRating(decimal newRating)
    {
        Rating = newRating;
        RaiseStatusChanged($"Рейтинг обновлен: {newRating}");
    }

    // Методы для вызова событий (исправление ошибки CS0070)
    protected virtual void RaiseOrderSubmitted(OrderEventArgs order)
    {
        OnOrderSubmitted?.Invoke(this, order);
    }

    protected virtual void RaiseQuoteProvided(decimal amount)
    {
        OnQuoteProvided?.Invoke(this, amount);
    }

    protected virtual void RaiseStatusChanged(string message)
    {
        OnSupplierStatusChanged?.Invoke(this, message);
    }
}

// Производные классы
public class Manufacturer : Supplier
{
    // Новые свойства
    public int ProductionCapacity { get; set; }
    public string FactoryLocation { get; set; }
    public bool HasResearchDepartment { get; set; }
    public List<string> ProductionTechnologies { get; set; }

    public int FoundedYear { get; set; }

    public Manufacturer(int supplierId, string companyName, string productType, int foundedYear)
        : base(supplierId, companyName, productType)
    {
        FoundedYear = foundedYear;
        ProductionCapacity = 10000;
        FactoryLocation = "Основное производство";
        HasResearchDepartment = true;
        ProductionTechnologies = new List<string>();
        MinimumOrderAmount = 5000m;
    }

    public override string ProvideQuote()
    {
        var quoteAmount = CalculateQuote();
        RaiseQuoteProvided(quoteAmount);
        return $"Компания {CompanyName} (основана в {FoundedYear}) предоставляет котировку на {ProductType}.";
    }

    // Новые методы
    public void AddProductionTechnology(string technology)
    {
        ProductionTechnologies.Add(technology);
        RaiseStatusChanged($"Добавлена технология производства: {technology}");
    }

    public string GetProductionInfo()
    {
        return $"Производственная мощность: {ProductionCapacity} единиц/мес, Локация: {FactoryLocation}";
    }

    public override decimal CalculateQuote()
    {
        var baseQuote = base.CalculateQuote();
        // Производители дают скидку за объем
        return ProductionCapacity > 50000 ? baseQuote * 0.9m : baseQuote;
    }

    public override void UpdateRating(decimal newRating)
    {
        // Производители имеют повышенный базовый рейтинг
        base.UpdateRating(Math.Min(newRating + 0.2m, 5.0m));
    }

    public override string SubmitOrder()
    {
        var order = CreateOrder();
        RaiseOrderSubmitted(order);
        return $"Производственный заказ отправлен в {CompanyName}.";
    }
}

public class Retailer : Supplier
{
    // Новые свойства
    public int StoreCount { get; set; }
    public decimal AverageMonthlyRevenue { get; set; }
    public bool HasOnlineStore { get; set; }
    public Dictionary<string, int> Inventory { get; set; }

    public string StoreLocation { get; set; }

    public Retailer(int supplierId, string companyName, string productType, string storeLocation)
        : base(supplierId, companyName, productType)
    {
        StoreLocation = storeLocation;
        StoreCount = 1;
        AverageMonthlyRevenue = 50000m;
        HasOnlineStore = false;
        Inventory = new Dictionary<string, int>();
        MinimumOrderAmount = 500m;
    }

    public override string SubmitOrder()
    {
        var order = CreateOrder();
        RaiseOrderSubmitted(order);
        return $"Заказ отправлен в компанию {CompanyName}, расположенную в {StoreLocation}.";
    }

    // Новые методы
    public void AddStore(string location)
    {
        StoreCount++;
        RaiseStatusChanged($"Открыт новый магазин в {location}");
    }

    public void UpdateInventory(string product, int quantity)
    {
        Inventory[product] = quantity;
        RaiseStatusChanged($"Обновлен инвентарь: {product} - {quantity} шт.");
    }

    public string GetInventorySummary()
    {
        return Inventory.Count > 0 
            ? $"Товаров в инвентаре: {Inventory.Count}, Общее количество: {Inventory.Values.Sum()}"
            : "Инвентарь пуст";
    }

    public override OrderEventArgs CreateOrder()
    {
        // Розничные продавцы имеют меньшие минимальные заказы
        return new OrderEventArgs(
            $"RET-{SupplierId}-{DateTime.Now:yyyyMMdd-HHmmss}",
            Math.Max(CalculateQuote(), 500m),
            DateTime.Now,
            $"Розничная продукция: {ProductType}"
        );
    }
}

public class Importer : Supplier
{
    // Новые свойства
    public List<string> SourceCountries { get; set; }
    public decimal ImportDuties { get; set; }
    public bool HasWarehouse { get; set; }
    public Dictionary<string, decimal> ShippingCosts { get; set; }

    public string OriginCountry { get; set; }

    public Importer(int supplierId, string companyName, string productType, string originCountry)
        : base(supplierId, companyName, productType)
    {
        OriginCountry = originCountry;
        SourceCountries = new List<string> { originCountry };
        ImportDuties = 0.1m; // 10% пошлина
        HasWarehouse = true;
        ShippingCosts = new Dictionary<string, decimal>();
        MinimumOrderAmount = 2000m;
    }

    public override string GetCompanyInfo()
    {
        return $"ID: {SupplierId}, Компания: {CompanyName}, Тип продукции: {ProductType}, Страна происхождения: {OriginCountry}";
    }

    // Новые методы
    public void AddSourceCountry(string country)
    {
        SourceCountries.Add(country);
        RaiseStatusChanged($"Добавлена страна-источник: {country}");
    }

    public decimal CalculateShippingCost(string destination)
    {
        return ShippingCosts.ContainsKey(destination) ? ShippingCosts[destination] : 100m;
    }

    public override decimal CalculateQuote()
    {
        var baseQuote = base.CalculateQuote();
        // Импортеры добавляют пошлины и стоимость доставки
        return baseQuote * (1 + ImportDuties) + 200m;
    }

    public string GetInternationalInfo()
    {
        return $"Страны-источники: {string.Join(", ", SourceCountries)}, Пошлины: {ImportDuties * 100}%";
    }

    public override string SubmitOrder()
    {
        var order = CreateOrder();
        RaiseOrderSubmitted(order);
        return $"Международный заказ отправлен в {CompanyName} из {OriginCountry}.";
    }
}

// Класс для управления поставщиками
public class SupplierManager
{
    private List<Supplier> suppliers = new List<Supplier>();
    
    // Делегаты для фильтрации
    public SupplierFilter ActiveFilter { get; set; }
    public SupplierFilter HighRatingFilter { get; set; }

    public event Action<Supplier> OnSupplierAdded;
    public event Action<string> OnManagementAction;

    public void AddSupplier(Supplier supplier)
    {
        suppliers.Add(supplier);
        
        // Подписка на события поставщика через лямбда-выражения
        supplier.OnOrderSubmitted += (s, order) => 
        {
            Console.WriteLine($"[МЕНЕДЖЕР] Заказ {order.OrderNumber} от {s.CompanyName} на сумму {order.OrderAmount:C}");
        };
        
        supplier.OnQuoteProvided += (s, amount) =>
        {
            Console.WriteLine($"[МЕНЕДЖЕР] Котировка от {s.CompanyName}: {amount:C}");
        };
        
        supplier.OnSupplierStatusChanged += (s, message) =>
        {
            Console.WriteLine($"[СТАТУС] : {message}");
        };

        OnSupplierAdded?.Invoke(supplier);
        OnManagementAction?.Invoke($"Добавлен поставщик: {supplier.CompanyName}");
    }

    public List<Supplier> GetFilteredSuppliers(SupplierFilter filter)
    {
        return suppliers.Where(s => filter(s)).ToList();
    }

    public void ProcessHighRatingSuppliers(Action<Supplier> action)
    {
        var highRated = GetFilteredSuppliers(HighRatingFilter);
        highRated.ForEach(action);
    }

    public decimal GetTotalBusinessVolume()
    {
        return suppliers.Sum(s => s.GetTotalRevenue());
    }

    public void DisplayAllSuppliers()
    {
        Console.WriteLine("\n=== ВСЕ ПОСТАВЩИКИ ===");
        suppliers.ForEach(s => 
        {
            Console.WriteLine(s.GetCompanyInfo());
            Console.WriteLine($"Заказов: {s.GetOrderCount()}, Выручка: {s.GetTotalRevenue():C}");
            Console.WriteLine();
        });
    }
}


        var manager = new SupplierManager();

        // Настройка фильтров через лямбда-выражения
        manager.ActiveFilter = s => s.IsActive && s.Rating >= 4.0m;
        manager.HighRatingFilter = s => s.Rating >= 4.5m;

        // Подписка на события менеджера (исправление ошибки CS1061)
        manager.OnSupplierAdded += supplier =>
        {
            Console.WriteLine($"[СИСТЕМА] Зарегистрирован новый поставщик: {supplier.CompanyName}");
        };

        manager.OnManagementAction += action =>
        {
            Console.WriteLine($"[УПРАВЛЕНИЕ] {action}");
        };

        // Создание поставщиков
        Manufacturer manufacturer = new Manufacturer(1, "TechProd", "Электроника", 1995);
        Retailer retailer = new Retailer(2, "ShopWorld", "Одежда", "Москва");
        Importer importer = new Importer(3, "GlobalTrade", "Фрукты", "Испания");

        // Добавление в менеджер
        manager.AddSupplier(manufacturer);
        manager.AddSupplier(retailer);
        manager.AddSupplier(importer);

        // Демонстрация новых методов
        Console.WriteLine("=== ДЕМОНСТРАЦИЯ НОВЫХ ВОЗМОЖНОСТЕЙ ===");

        // Добавление сертификаций и технологий
        manufacturer.AddCertification("ISO 9001");
        manufacturer.AddProductionTechnology("Автоматическая сборка");
        
        retailer.AddCertification("Quality Retail");
        retailer.UpdateInventory("Футболки", 100);
        retailer.AddStore("Санкт-Петербург");
        
        importer.AddCertification("Organic Certified");
        importer.AddSourceCountry("Италия");

        // Выполнение заказов и котировок
        Console.WriteLine("\n=== ВЫПОЛНЕНИЕ ОПЕРАЦИЙ ===");
        Console.WriteLine(manufacturer.SubmitOrder());
        Console.WriteLine(retailer.ProvideQuote());
        Console.WriteLine(importer.SubmitOrder());

        // Обновление рейтингов
        manufacturer.UpdateRating(4.8m);
        retailer.UpdateRating(4.3m);
        importer.UpdateRating(4.9m);

        // Использование фильтров
        Console.WriteLine("\n=== ФИЛЬТРАЦИЯ ПОСТАВЩИКОВ ===");
        var activeSuppliers = manager.GetFilteredSuppliers(manager.ActiveFilter);
        Console.WriteLine("Активные поставщики с высоким рейтингом:");
        activeSuppliers.ForEach(s => Console.WriteLine($"- {s.CompanyName} (Рейтинг: {s.Rating})"));

        // Групповые операции
        Console.WriteLine("\n=== ГРУППОВЫЕ ОПЕРАЦИИ ===");
        manager.ProcessHighRatingSuppliers(s => 
        {
            Console.WriteLine($"Высокорейтинговый: {s.CompanyName}");
            Console.WriteLine(s.SubmitOrder());
        });

        // Отображение всей информации
        manager.DisplayAllSuppliers();

        // Демонстрация специализированных методов
        Console.WriteLine("=== СПЕЦИАЛИЗИРОВАННАЯ ИНФОРМАЦИЯ ===");
        Console.WriteLine(manufacturer.GetProductionInfo());
        Console.WriteLine(retailer.GetInventorySummary());
        Console.WriteLine(importer.GetInternationalInfo());

        Console.WriteLine($"\nОбщий объем бизнеса: {manager.GetTotalBusinessVolume():C}");

        // Дополнительные операции для демонстрации коллекций
        Console.WriteLine("\n=== РАБОТА С КОЛЛЕКЦИЯМИ ===");
        retailer.UpdateInventory("Джинсы", 50);
        retailer.UpdateInventory("Куртки", 30);
        Console.WriteLine(retailer.GetInventorySummary());

        importer.ShippingCosts.Add("Москва", 150m);
        importer.ShippingCosts.Add("Санкт-Петербург", 200m);
        Console.WriteLine($"Стоимость доставки в Москву: {importer.CalculateShippingCost("Москва"):C}");


[СИСТЕМА] Зарегистрирован новый поставщик: TechProd
[УПРАВЛЕНИЕ] Добавлен поставщик: TechProd
[СИСТЕМА] Зарегистрирован новый поставщик: ShopWorld
[УПРАВЛЕНИЕ] Добавлен поставщик: ShopWorld
[СИСТЕМА] Зарегистрирован новый поставщик: GlobalTrade
[УПРАВЛЕНИЕ] Добавлен поставщик: GlobalTrade
=== ДЕМОНСТРАЦИЯ НОВЫХ ВОЗМОЖНОСТЕЙ ===
[СТАТУС] : Добавлена сертификация: ISO 9001
[СТАТУС] : Добавлена технология производства: Автоматическая сборка
[СТАТУС] : Добавлена сертификация: Quality Retail
[СТАТУС] : Обновлен инвентарь: Футболки - 100 шт.
[СТАТУС] : Открыт новый магазин в Санкт-Петербург
[СТАТУС] : Добавлена сертификация: Organic Certified
[СТАТУС] : Добавлена страна-источник: Италия

=== ВЫПОЛНЕНИЕ ОПЕРАЦИЙ ===
[МЕНЕДЖЕР] Заказ ORD-1-20251130-141332 от TechProd на сумму ¤5,500.00
Производственный заказ отправлен в TechProd.
[МЕНЕДЖЕР] Котировка от ShopWorld: ¤550.00
Компания ShopWorld предоставляет котировку на продукцию типа Одежда.
[МЕНЕДЖЕР] Заказ ORD-3-20251130-141332 от GlobalTrade 